<a href="https://colab.research.google.com/github/RRADJon/TEMPO/blob/main/Boltz2_Nvidia_NIM_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Make sure to visit https://build.nvidia.com/mit/boltz and select "get API key"
include this API key in your secrets tab under NVIDIA_API_KEY and turn on notebook access.

In [30]:
#@title Setup call to NVIDIA NIM Boltz-2 and Run
!pip install -q httpx fastapi nest_asyncio

import asyncio
import json
import logging
import sys
import os
from typing import Any, Dict, Optional
from pathlib import Path
import httpx
from fastapi import HTTPException
from google.colab import userdata
import nest_asyncio

# Apply nest_asyncio to allow asyncio.run() to work in a notebook
nest_asyncio.apply()

# 2. Setup Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 3. Retrieve the Secret Key
try:
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
except Exception:
    print("Error: Make sure you added 'NVIDIA_API_KEY' to the Secrets tab and enabled Notebook access.")
    sys.exit(1)

STATUS_URL = "https://api.nvcf.nvidia.com/v2/nvcf/pexec/status/{task_id}"
PUBLIC_URL = "https://health.api.nvidia.com/v1/biology/mit/boltz2/predict"

async def make_nvcf_call(function_url: str,
                        data: Dict[str, Any],
                        additional_headers: Optional[Dict[str, Any]] = None,
                        NVCF_POLL_SECONDS: int = 300,
                        MANUAL_TIMEOUT_SECONDS: int = 400) -> Dict:

    async with httpx.AsyncClient() as client:
        # We inject the actual API key here
        headers = {
            "Authorization": f"Bearer {NVIDIA_API_KEY}",
            "NVCF-POLL-SECONDS": f"{NVCF_POLL_SECONDS}",
            "Content-Type": "application/json"
        }
        if additional_headers:
            headers.update(additional_headers)

        response = await client.post(function_url,
                                     json=data,
                                     headers=headers,
                                     timeout=MANUAL_TIMEOUT_SECONDS)

        if response.status_code == 202:
            task_id = response.headers.get("nvcf-reqid")
            while True:
                # Poll for status
                status_response = await client.get(STATUS_URL.format(task_id=task_id),
                                                   headers=headers,
                                                   timeout=MANUAL_TIMEOUT_SECONDS)
                if status_response.status_code == 200:
                    return status_response.status_code, status_response
                elif status_response.status_code in [400, 401, 404, 422, 500]:
                    raise HTTPException(status_code=status_response.status_code, detail=status_response.text)
                await asyncio.sleep(5) # Add a small sleep to avoid spamming the poll
        elif response.status_code == 200:
            return response.status_code, response
        else:
            raise HTTPException(status_code=response.status_code, detail=response.text)

async def main():
    sequence = "MTEYKLVVVGACGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKE"  #@param {type:"string"}
    ligand = "CCC(=O)N1CCN2Cc3cc(C#C)c(c(F)c3OC[C@H]2C1)c4c(O)cccc4Cl"#@param {type:"string"}
    output_file = Path("output.json")

    data = {
        "polymers": [
            {
                "id": "A",
                "molecule_type": "protein",
                "sequence": sequence,
                "msa": {
                    "uniref90": {
                        "a3m": {
                            "alignment": f">seq1\n{sequence}",
                            "format": "a3m"
                        }
                    }
                }
            }
        ],
        "ligands": [
            {
                "smiles": ligand,
                "id": "L1",
                "predict_affinity" : True
            }
        ],
        "recycling_steps": 1,
        "sampling_steps": 50,
        "diffusion_samples": 3,
        "step_scale": 1.2,
        "without_potentials": True
    }

    print("Sending request to NVIDIA Boltz-2...")
    try:
        code, response = await make_nvcf_call(function_url=PUBLIC_URL, data=data)

        if code == 200:
            print(f"Success! Status Code: {code}")
            response_dict = response.json()
            output_file.write_text(json.dumps(response_dict, indent=4))

            print(f"Structures found: {len(response_dict['structures'])}")
            print(f"Confidence scores: {response_dict['confidence_scores']}")
            print(f"Results saved to: {output_file.absolute()}")
    except Exception as e:
        print(f"An error occurred: {e}")

# In Colab, we can just call main() using the applied nest_asyncio
if __name__ == "__main__":
    asyncio.run(main())

Sending request to NVIDIA Boltz-2...
Success! Status Code: 200
Structures found: 3
Confidence scores: [0.8858441710472107, 0.890402615070343, 0.8662800192832947]
Results saved to: /content/output.json


In [31]:
#@title Visualization
import json
import py3Dmol
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

def analyze_boltz_results(json_path="output.json"):
    if not Path(json_path).exists():
        print(f"Error: {json_path} not found.")
        return

    with open(json_path, 'r') as f:
        data = json.load(f)

    # --- 1. 3D Structure Visualization ---
    if 'structures' in data and len(data['structures']) > 0:
        struct = data['structures'][0]
        structure_data = struct.get('structure', '')
        fmt = struct.get('format', 'pdb')

        print(f"--- Visualizing Top Ranked Structure ({fmt.upper()}) ---")
        viewer = py3Dmol.view(width=800, height=400)
        viewer.addModel(structure_data, fmt)
        viewer.setStyle({'cartoon': {'color': 'spectrum'}})
        # Highlight the ligand (L1)
        viewer.setStyle({'resn': 'LIG1'}, {'stick': {'colorscheme': 'magentaCarbon'}})
        viewer.zoomTo()
        viewer.show()

    # --- 3. Affinity & IC50 Prediction Table ---
    # Note: Using 'affinity_predictions' based on your previous output snippet
    affinity_root = data.get('affinity_predictions', data.get('affinities', {}))
    affinity_data = affinity_root.get('L1', {})

    if affinity_data:
        # pIC50 Calculation
        pIC50 = affinity_data.get('affinity_pic50', [0])[0]
        ic50_um = 10**(6 - pIC50)

        # Binary probability (is this a binder or not?)
        bind_prob = affinity_data.get('affinity_probability_binary', [0])[0]

        # 3. Create the Consolidated Metrics Table
        metrics = {
            "Metric": [
                "pIC50 (Log Scale)",
                "Predicted IC50",
                "Binding Probability"
            ],
            "Value": [
                f"{pIC50:.4f}",
                f"{ic50_um:.3f} µM",
                f"{bind_prob:.2%}"
            ]
        }

        print("\n### Boltz-2 Combined Prediction Results")
        df_metrics = pd.DataFrame(metrics)
        display(HTML(df_metrics.to_html(index=False, classes='table table-striped')))
analyze_boltz_results()

--- Visualizing Top Ranked Structure (MMCIF) ---


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


### Boltz-2 Combined Prediction Results


Metric,Value
pIC50 (Log Scale),7.4722
Predicted IC50,0.034 µM
Binding Probability,57.04%
